In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score

pd.set_option('display.max_columns', None)

In [2]:
df_clean = pd.read_pickle('data/clean_transactions.pkl')
print(df_clean.shape)

# The last date in the data. Everything is measured relative to it.
max_date = df_clean['InvoiceDate'].max()
print(max_date)                        

HORIZON = 90        # "will the customer buy again within 90 days?"
QUIET_DAYS = 60     # a customer is "quiet" after 60 days without a purchase (see Part C)

(805549, 9)
2011-12-09 12:50:00


**Helper Function**

In [3]:
def build_dataset(df, cutoff, horizon_days):
    past = df[df['InvoiceDate'] <= cutoff]
    future = df[(df['InvoiceDate'] > cutoff) &
                (df['InvoiceDate'] <= cutoff + pd.Timedelta(days=horizon_days))]
    feat = past.groupby('Customer ID').agg(
        Recency=('InvoiceDate', lambda x: (cutoff - x.max()).days),
        Frequency=('Invoice', 'nunique'),
        Monetary=('TotalPrice', 'sum'),
        FirstPurchase=('InvoiceDate', 'min')).reset_index()
    feat['label'] = feat['Customer ID'].isin(set(future['Customer ID'])).astype(int)
    return feat

def prep(feat, cap=None):
    spend = feat['Monetary'].clip(upper=cap) if cap else feat['Monetary']
    return pd.DataFrame({'Recency': feat['Recency'],
                         'log_Frequency': np.log1p(feat['Frequency']),
                         'log_Monetary': np.log1p(spend)})

**Out-of-time test**

In [4]:
def run(horizon=90, winsorize=False):
    test_cutoff = max_date - pd.Timedelta(days=horizon)
    train_cutoff = test_cutoff - pd.Timedelta(days=horizon)
    train = build_dataset(df_clean, train_cutoff, horizon)
    test = build_dataset(df_clean, test_cutoff, horizon)
    cap = train['Monetary'].quantile(0.99) if winsorize else None
    X_train, X_test = prep(train, cap), prep(test, cap)
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(random_state=42).fit(scaler.transform(X_train), train['label'])
    test['p'] = model.predict_proba(scaler.transform(X_test))[:, 1]
    pred = (test['p'] >= 0.5).astype(int)
    naive = (test['Recency'] <= train['Recency'].median()).astype(int)   # "recent buyers will return"
    y = test['label']
    res = {'window': horizon,
           'train_rate': round(train['label'].mean(), 3), 'test_rate': round(y.mean(), 3),
           'model_acc': round(accuracy_score(y, pred), 3),
           'naive_acc': round(accuracy_score(y, naive), 3),
           'majority_acc': round(1 - y.mean(), 3),
           'model_auc': round(roc_auc_score(y, test['p']), 3),
           'naive_auc': round(roc_auc_score(y, -test['Recency']), 3),
           'precision': round(precision_score(y, pred), 3),
           'recall': round(recall_score(y, pred), 3)}
    return res, test

In [5]:
# B1. Main result (90-day window)
res90, test90 = run(90)
res90

{'window': 90,
 'train_rate': np.float64(0.321),
 'test_rate': np.float64(0.434),
 'model_acc': 0.709,
 'naive_acc': 0.704,
 'majority_acc': np.float64(0.566),
 'model_auc': 0.793,
 'naive_auc': 0.763,
 'precision': 0.786,
 'recall': 0.454}

In [6]:
# B2. Do the conclusions change with a 60-, 90-, or 120-day window?
pd.DataFrame([run(h)[0] for h in (60, 90, 120)])

,window,train_rate,test_rate,model_acc,naive_acc,majority_acc,model_auc,naive_auc,precision,recall
0,60,0.293,0.369,0.732,0.684,0.631,0.784,0.750,0.719,0.449
1,90,0.321,0.434,0.709,0.704,0.566,0.793,0.763,0.786,0.454
2,120,0.393,0.468,0.724,0.715,0.532,0.807,0.777,0.784,0.566


In [7]:
# B3. Outlier check: cap spend at the training set's 99th percentile. Do results move?
run(90, winsorize=True)[0]

{'window': 90,
 'train_rate': np.float64(0.321),
 'test_rate': np.float64(0.434),
 'model_acc': 0.709,
 'naive_acc': 0.704,
 'majority_acc': np.float64(0.566),
 'model_auc': 0.793,
 'naive_auc': 0.763,
 'precision': 0.785,
 'recall': 0.455}

In [8]:
# B4. Does a higher score mean a higher repeat rate? (decile 10 = highest scores)
test90['decile'] = pd.qcut(test90['p'].rank(method='first'), 10, labels=range(1, 11))
print(test90.groupby('decile', observed=True)['label'].agg(['count', 'mean']).round(3))

k = int(len(test90) * 0.10)
print('Repeat rate, top 10% by model score:      ', round(test90.nlargest(k, 'p')['label'].mean(), 3))
print('Repeat rate, 10% most recent buyers only: ', round(test90.nsmallest(k, 'Recency')['label'].mean(), 3))

        count   mean
decile              
1         529  0.079
2         528  0.169
3         528  0.220
4         528  0.292
5         528  0.316
6         528  0.483
7         527  0.514
8         529  0.609
9         528  0.748
10        528  0.911
Repeat rate, top 10% by model score:       0.911
Repeat rate, 10% most recent buyers only:  0.777


## What counts as "gone quiet"?

In [9]:
inv = (df_clean.groupby(['Customer ID', 'Invoice'])['InvoiceDate'].min()
       .reset_index().sort_values(['Customer ID', 'InvoiceDate']))
gaps = inv.groupby('Customer ID')['InvoiceDate'].diff().dt.days.dropna()
print('median gap:', gaps.median(), '| 75th percentile:', gaps.quantile(0.75)) 

median gap: 24.0 | 75th percentile: 61.0


## Train the final model and score every customer

In [10]:
train_final = build_dataset(df_clean, max_date - pd.Timedelta(days=HORIZON), HORIZON)
X_f = prep(train_final)
scaler_f = StandardScaler().fit(X_f)
model_f = LogisticRegression(random_state=42).fit(scaler_f.transform(X_f), train_final['label'])
print(dict(zip(X_f.columns, model_f.coef_[0].round(3))))   # Recency negative; Frequency, Monetary positive

scored = build_dataset(df_clean, max_date, HORIZON).drop(columns='label')
scored['p_return'] = model_f.predict_proba(scaler_f.transform(prep(scored)))[:, 1]
print(len(scored), round(scored['p_return'].mean(), 3)) 

{'Recency': np.float64(-0.748), 'log_Frequency': np.float64(0.468), 'log_Monetary': np.float64(0.304)}
5878 0.459


## Rank by value at risk

In [11]:
def rfm_segments(d):
    d = d.copy()
    d['R_score'] = pd.qcut(d['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
    d['F_score'] = pd.qcut(d['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    d['M_score'] = pd.qcut(d['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
    def seg(row):
        r, f, m = row['R_score'], row['F_score'], row['M_score']
        if r >= 4 and f >= 4 and m >= 4: return 'Champions'
        elif r >= 3 and f >= 3: return 'Loyal Customers'
        elif r >= 4 and f <= 2: return 'New Customers'
        elif r <= 2 and f >= 4 and m >= 4: return 'At Risk (High Value)'
        elif r <= 2 and f >= 3: return 'At Risk'
        elif r <= 2 and f <= 2 and m <= 2: return 'Lost'
        else: return 'Needs Attention'
    d['Segment'] = d.apply(seg, axis=1)
    return d

def add_value_at_risk(d, snapshot, p_col):
    d = d.copy()
    tenure = (snapshot - d['FirstPurchase']).dt.days.clip(lower=90)
    d['spend_per_90d'] = d['Monetary'] / (tenure / 90)
    d['value_at_risk'] = (1 - d[p_col]) * d['spend_per_90d']
    return d

In [12]:
scored = add_value_at_risk(scored, max_date, 'p_return')
scored = rfm_segments(scored)

scored['eligible'] = scored['Recency'] >= QUIET_DAYS
el = scored['eligible']
n_el = int(el.sum())
print('eligible:', n_el, 'of', len(scored))               

scored['priority_rank'] = np.nan
scored.loc[el, 'priority_rank'] = scored.loc[el, 'value_at_risk'].rank(ascending=False, method='first')

scored['priority_tier'] = 'Active - no contact'
scored.loc[el, 'priority_tier'] = pd.cut(
    scored.loc[el, 'priority_rank'], bins=[0, .10 * n_el, .30 * n_el, .60 * n_el, n_el],
    labels=['1 - Target first', '2 - Consider', '3 - Low priority', '4 - No offer']).astype(str)

eligible: 3482 of 5878


In [13]:
tier_order = ['1 - Target first', '2 - Consider', '3 - Low priority', '4 - No offer', 'Active - no contact']
print(scored.groupby('priority_tier').agg(
    customers=('Customer ID', 'count'), avg_p_return=('p_return', 'mean'),
    avg_spend_per_90d=('spend_per_90d', 'mean'), total_value_at_risk=('value_at_risk', 'sum')
).reindex(tier_order).round(2))
print()
print(pd.crosstab(scored['Segment'], scored['priority_tier'])[tier_order])
print()
cols = ['Customer ID', 'Segment', 'Recency', 'Frequency', 'Monetary', 'p_return', 'spend_per_90d', 'value_at_risk']
scored.sort_values('priority_rank').head(8)[cols].round(2)

                     customers  avg_p_return  avg_spend_per_90d  \
priority_tier                                                     
1 - Target first           348          0.45            1026.18   
2 - Consider               696          0.42             337.95   
3 - Low priority          1045          0.35             161.60   
4 - No offer              1393          0.17              48.14   
Active - no contact       2396          0.69            1013.52   

                     total_value_at_risk  
priority_tier                             
1 - Target first               159856.80  
2 - Consider                   114372.85  
3 - Low priority                95464.35  
4 - No offer                    50729.26  
Active - no contact            439142.90  

priority_tier         1 - Target first  2 - Consider  3 - Low priority  \
Segment                                                                  
At Risk                             32           124               270   
At Ris

,Customer ID,Segment,Recency,Frequency,Monetary,p_return,spend_per_90d,value_at_risk
2722,15098,Loyal Customers,181,3,39916.50,0.63,19847.98,7288.39
3371,15749,At Risk,234,3,44534.30,0.58,12109.02,5029.29
1538,13902,At Risk (High Value),631,5,34095.26,0.24,4238.36,3236.60
0,12346,At Risk (High Value),325,12,77556.46,0.69,9627.70,3029.59
4364,16754,At Risk (High Value),371,29,67502.47,0.74,9477.73,2458.47
243,12590,Needs Attention,210,2,9864.26,0.48,4227.54,2178.75
404,12752,Needs Attention,80,1,4366.78,0.52,4366.78,2105.13
341,12688,Needs Attention,113,1,4873.81,0.49,3881.80,1984.52


## Backtest: would this ranking have worked?

In [14]:
test_cutoff = max_date - pd.Timedelta(days=HORIZON)
train_cutoff = test_cutoff - pd.Timedelta(days=HORIZON)
bt_train = build_dataset(df_clean, train_cutoff, HORIZON)
bt = build_dataset(df_clean, test_cutoff, HORIZON)
sc_b = StandardScaler().fit(prep(bt_train))
m_b = LogisticRegression(random_state=42).fit(sc_b.transform(prep(bt_train)), bt_train['label'])
bt['p'] = m_b.predict_proba(sc_b.transform(prep(bt)))[:, 1]
bt = add_value_at_risk(bt, test_cutoff, 'p')
bt['lapsed'] = 1 - bt['label']
bt['lost_value'] = bt['lapsed'] * bt['spend_per_90d']
bt = bt[bt['Recency'] >= QUIET_DAYS]                       # quiet customers only

print('eligible:', len(bt), '| lapse rate:', round(bt['lapsed'].mean(), 3))    # expect 3788 and 0.681
rows = []
for frac in (0.10, 0.20):
    k = int(len(bt) * frac)
    for name, col in [('value at risk', 'value_at_risk'), ('spend only', 'spend_per_90d')]:
        top = bt.nlargest(k, col)
        rows.append({'targeted': f'top {int(frac * 100)}%', 'ranked by': name,
                     'lost value captured %': round(top['lost_value'].sum() / bt['lost_value'].sum() * 100, 1),
                     'actually lapsed %': round(top['lapsed'].mean() * 100, 1)})
pd.DataFrame(rows)

eligible: 3788 | lapse rate: 0.681


,targeted,ranked by,lost value captured %,actually lapsed %
0,top 10%,value at risk,46.1,54.2
1,top 10%,spend only,43.8,42.9
2,top 20%,value at risk,58.5,52.2
3,top 20%,spend only,56.5,45.0


## Export for the dashboard

In [15]:
os.makedirs('data', exist_ok=True)
export_cols = ['Customer ID', 'Segment', 'priority_tier', 'priority_rank', 'eligible', 'Recency',
               'Frequency', 'Monetary', 'p_return', 'spend_per_90d', 'value_at_risk']
scored[export_cols].round(3).to_csv('data/scored_customers.csv', index=False)
print('saved', len(scored), 'rows')

saved 5878 rows
